# Pandas Data Cleaning Practice
## Overview
This notebook contains practical exercises for working with Pandas DataFrames and performing common data cleaning operations.

## Objectives
- Create and inspect DataFrames
- Identify and handle missing values
- Remove duplicate records
- Convert data types
- Clean and standardize data
- Filter and analyze data using Pandas

In [1]:
## 1. Import Required Libraries
import pandas as pd
import numpy as np

In [2]:
print(pd.__version__)

2.3.3


## Build a Deliberately Messy Sales Dataset
To practice real-world data cleaning, I will create a small sales dataset by hand.
The dataset intentionally contains:
- Missing values in different columns
- Inconsistent category casing
- Different types of information that require different cleaning decisions
The goal is to diagnose these problems first and clean them only after understanding what is wrong.

In [3]:
data = {
    "date": [
        "2026-08-01",
        "2026-08-02",
        None,
        "2026-08-04",
        "2026-08-05",
        "2026-08-06",
        "2026-08-07",
        "2026-08-08"
    ],
    "category": [
        "Electronics",
        "electronics",
        "Furniture",
        None,
        "Clothing",
        "Electronics",
        "furniture",
        "Clothing"
    ],
    "amount": [
        250.0,
        180.0,
        None,
        450.0,
        120.0,
        320.0,
        275.0,
        None
    ],
    "customer_age": [
        25,
        None,
        34,
        41,
        29,
        36,
        None,
        31
    ]
}

df = pd.DataFrame(data)

df

,date,category,amount,customer_age
0,2026-08-01,Electronics,250.0,25.0
1,2026-08-02,electronics,180.0,NaN
2,None,Furniture,NaN,34.0
3,2026-08-04,None,450.0,41.0
4,2026-08-05,Clothing,120.0,29.0
5,2026-08-06,Electronics,320.0,36.0
6,2026-08-07,furniture,275.0,NaN
7,2026-08-08,Clothing,NaN,31.0


### Initial Observation
The dataset contains deliberate data-quality problems. There are missing values in the `date`, `category`, `amount`, and `customer_age` columns. The `category` column also contains inconsistent capitalization, such as `"Electronics"` and `"electronics"`.
I will diagnose these issues quantitatively before making any changes.

## Diagnose the Dataset Before Cleaning
Before changing any values, I will inspect the dataset using `head()`, `info()`, `shape`, `describe()`, and `isna().sum()`.
This helps establish the condition of the original dataset before cleaning.

In [4]:
df.head()

,date,category,amount,customer_age
0,2026-08-01,Electronics,250.0,25.0
1,2026-08-02,electronics,180.0,NaN
2,None,Furniture,NaN,34.0
3,2026-08-04,None,450.0,41.0
4,2026-08-05,Clothing,120.0,29.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          7 non-null      object 
 1   category      7 non-null      object 
 2   amount        6 non-null      float64
 3   customer_age  6 non-null      float64
dtypes: float64(2), object(2)
memory usage: 384.0+ bytes


In [6]:
df.shape

(8, 4)

In [7]:
df.describe()

,amount,customer_age
count,6.000000,6.000000
mean,265.833333,32.666667
std,114.735202,5.609516
min,120.000000,25.000000
25%,197.500000,29.500000
50%,262.500000,32.500000
75%,308.750000,35.500000
max,450.000000,41.000000


In [8]:
df.isna().sum()

date            1
category        1
amount          2
customer_age    2
dtype: int64

### Diagnosis Findings
The dataset contains 8 rows and 4 columns.
The `date` column contains missing data, the `category` column contains one missing value, the `amount` column contains missing values, and the `customer_age` column also contains missing values.
The `category` column additionally contains inconsistent capitalization. For example, `"Electronics"` and `"electronics"` represent the same category but are currently treated as different values.
No cleaning operation was performed before this diagnosis, so these observations describe the original messy dataset.

## Missing Data Strategy

Missing values should not automatically be removed from the entire dataset. Each column needs a strategy based on the meaning of its data.
- **date:** I will use `dropna()` for the row with a missing date because the transaction date is important for identifying when a sale occurred. Creating an artificial date could introduce incorrect information.
- **category:** I will use `fillna()` with `"Unknown"` because the transaction may still be useful even if its category is unavailable.
- **amount:** I will use `fillna()` with the median amount because the transaction should be retained and the median is less affected by unusually large or small sales.
- **customer_age:** I will use `fillna()` with the median customer age because age is numeric and the median provides a reasonable central value without removing the transaction.

In [9]:
# Drop rows where the essential transaction date is missing
df = df.dropna(subset=["date"]).copy()

# Fill missing category values
df["category"] = df["category"].fillna("Unknown")

# Fill missing amounts with the median
df["amount"] = df["amount"].fillna(df["amount"].median())

# Fill missing customer ages with the median
df["customer_age"] = df["customer_age"].fillna(df["customer_age"].median())

df

,date,category,amount,customer_age
0,2026-08-01,Electronics,250.0,25.0
1,2026-08-02,electronics,180.0,31.0
3,2026-08-04,Unknown,450.0,41.0
4,2026-08-05,Clothing,120.0,29.0
5,2026-08-06,Electronics,320.0,36.0
6,2026-08-07,furniture,275.0,31.0
7,2026-08-08,Clothing,262.5,31.0


### Verify Missing Values After Cleaning

The missing values have now been handled according to a column-specific strategy rather than applying one blanket operation to the entire DataFrame.

In [10]:
df.isna().sum()

date            0
category        0
amount          0
customer_age    0
dtype: int64

## `.loc` vs `.iloc`

Pandas provides two important ways to select data:
- `.loc[]` selects data using row and column labels.
- `.iloc[]` selects data using integer positions.
On a DataFrame with a default index, they may appear to behave similarly. However, after sorting the DataFrame, the index labels remain attached to their original rows while their positions change.

In [11]:
df = df.reset_index(drop=True)
df

,date,category,amount,customer_age
0,2026-08-01,Electronics,250.0,25.0
1,2026-08-02,electronics,180.0,31.0
2,2026-08-04,Unknown,450.0,41.0
3,2026-08-05,Clothing,120.0,29.0
4,2026-08-06,Electronics,320.0,36.0
5,2026-08-07,furniture,275.0,31.0
6,2026-08-08,Clothing,262.5,31.0


### Selecting the Same Row in the Original Order
In the original DataFrame, index label `2` is also at integer position `2`. Therefore, `.loc[2]` and `.iloc[2]` should return the same row.

In [12]:
print("Using .loc[2]:")
display(df.loc[2])
print("Using .iloc[2]:")
display(df.iloc[2])

Using .loc[2]:


date            2026-08-04
category           Unknown
amount               450.0
customer_age          41.0
Name: 2, dtype: object

Using .iloc[2]:


date            2026-08-04
category           Unknown
amount               450.0
customer_age          41.0
Name: 2, dtype: object

Both selections return the same record because the row label and integer position are currently the same.

### Comparing `.loc` and `.iloc` After Sorting

Now I will sort the DataFrame by `amount` in descending order.

Sorting changes the physical row positions but preserves the original index labels. Therefore, `.loc[2]` and `.iloc[2]` can now refer to different rows.

In [13]:
sorted_df = df.sort_values("amount", ascending=False)

sorted_df

,date,category,amount,customer_age
2,2026-08-04,Unknown,450.0,41.0
4,2026-08-06,Electronics,320.0,36.0
5,2026-08-07,furniture,275.0,31.0
6,2026-08-08,Clothing,262.5,31.0
0,2026-08-01,Electronics,250.0,25.0
1,2026-08-02,electronics,180.0,31.0
3,2026-08-05,Clothing,120.0,29.0


In [14]:
print("Using .loc[2]:")
display(sorted_df.loc[2])

print("Using .iloc[2]:")
display(sorted_df.iloc[2])

Using .loc[2]:


date            2026-08-04
category           Unknown
amount               450.0
customer_age          41.0
Name: 2, dtype: object

Using .iloc[2]:


date            2026-08-07
category         furniture
amount               275.0
customer_age          31.0
Name: 5, dtype: object

### Observation
After sorting, `.loc[2]` selects the row whose index label is `2`, while `.iloc[2]` selects the row currently located at integer position `2`.
    
They return different records because sorting changed the row positions but did not change the original index labels.

This demonstrates why `.loc` and `.iloc` should not be confused.

##  Boolean Filtering with Multiple Conditions
Boolean filtering in Pandas is similar to Boolean masking in NumPy.

I will filter the sales DataFrame using two conditions:
- The sale amount must be greater than 200.
- The category must be electronics.

Pandas uses `&` for element-wise AND operations. Each condition must be enclosed in parentheses.

In [15]:
electronics_high_value = df[
    (df["amount"] > 200) &
    (df["category"] == "Electronics")
]

electronics_high_value

,date,category,amount,customer_age
0,2026-08-01,Electronics,250.0,25.0
4,2026-08-06,Electronics,320.0,36.0


### Why `and` Does Not Work Like `&`

Python's `and` operator expects a single Boolean value.

A Pandas Series contains multiple Boolean values, so Pandas cannot use the regular `and` operator to combine Series element by element.

I will deliberately use `and` to observe the error and understand why `&` is required.

In [16]:
try:
    wrong_filter = df[
        (df["amount"] > 200) and
        (df["category"] == "Electronics")
    ]
except Exception as e:
    print(type(e).__name__)
    print(e)

ValueError
The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


### Observation
The incorrect filter raises a `ValueError` because a Pandas Series contains multiple Boolean values and cannot be evaluated as one single True or False value.

For element-wise conditions, Pandas requires `&` for AND and `|` for OR.

The correct expression is therefore:

`(condition_1) & (condition_2)`

## Detecting Inconsistent Categories

The dataset contains category values with inconsistent capitalization.

For example, `"Electronics"` and `"electronics"` represent the same category but Pandas treats them as different values.

I will use `value_counts()` to identify this data-quality issue before fixing it.

In [17]:
df["category"].value_counts()

category
Electronics    2
Clothing       2
electronics    1
Unknown        1
furniture      1
Name: count, dtype: int64

### Standardizing Category Names
The `value_counts()` result showed that capitalization caused the same category to appear as separate values.

For example, `"Electronics"` and `"electronics"` represent the same category.

I will standardize the entire column using a vectorized string operation instead of manually changing individual rows.

In [18]:
df["category"] = df["category"].str.strip().str.lower()

df["category"].value_counts()

category
electronics    3
clothing       2
unknown        1
furniture      1
Name: count, dtype: int64


The category values are now standardized using a vectorized string operation.

Values that differed only in capitalization are now treated as the same category. No individual rows were manually edited.

In [19]:
df

,date,category,amount,customer_age
0,2026-08-01,electronics,250.0,25.0
1,2026-08-02,electronics,180.0,31.0
2,2026-08-04,unknown,450.0,41.0
3,2026-08-05,clothing,120.0,29.0
4,2026-08-06,electronics,320.0,36.0
5,2026-08-07,furniture,275.0,31.0
6,2026-08-08,clothing,262.5,31.0


## GroupBy Analysis
The category column is now clean, so I can perform category-level analysis.

The `groupby()` operation follows the split-apply-combine approach: the DataFrame is split into categories, an aggregation is calculated for each category, and the results are combined into a summary.

### Mean Sales Amount per Category

First, I will calculate the average transaction amount for each category.

In [20]:
mean_amount = df.groupby("category")["amount"].mean()

mean_amount

category
clothing       191.25
electronics    250.00
furniture      275.00
unknown        450.00
Name: amount, dtype: float64

### Transaction Count per Category

Next, I will calculate how many transactions belong to each category.

In [21]:
transaction_count = df.groupby("category").size()

transaction_count

category
clothing       2
electronics    3
furniture      1
unknown        1
dtype: int64

### Total Sales Amount per Category

Finally, I will calculate the total sales amount for each category and sort the result in descending order.

The first category in the result will therefore be the category with the highest total sales amount.

In [22]:
total_amount = (
    df.groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

total_amount

category
electronics    750.0
unknown        450.0
clothing       382.5
furniture      275.0
Name: amount, dtype: float64


The sorted result shows the categories ranked by their total sales amount.

The category at the top of the result has generated the highest total sales amount in this cleaned dataset.

## Vectorized Operations vs `.apply()`

Pandas is designed to perform operations on entire columns efficiently.

When a calculation can be expressed directly as a column operation, a vectorized approach is generally preferred over `.apply()` because it avoids calling a Python function separately for every row.

I will perform the same calculation using both approaches and compare their performance with `%timeit`.

### Vectorized Calculation

First, I will calculate a new column using a direct vectorized expression.

The calculation adds a 10% increase to the transaction amount.

In [23]:
df["total_vectorized"] = df["amount"] * 1.10

df[["amount", "total_vectorized"]]

,amount,total_vectorized
0,250.0,275.00
1,180.0,198.00
2,450.0,495.00
3,120.0,132.00
4,320.0,352.00
5,275.0,302.50
6,262.5,288.75


### Create a Larger Synthetic Dataset

To make the performance comparison meaningful, I will create a larger synthetic dataset by repeating the cleaned sample data.

The resulting DataFrame will contain hundreds of thousands of rows.

In [24]:
large_df = pd.concat([df] * 50000, ignore_index=True)

large_df.shape

(350000, 5)

### Timing the Vectorized Operation

The following `%timeit` benchmark measures how long the vectorized calculation takes on the large dataset.

In [25]:
%timeit large_df["amount"] * 1.10

277 μs ± 20.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Timing the `.apply()` Operation

Now I will perform the same calculation using `.apply()` with a Python lambda function.

This allows a direct performance comparison with the vectorized approach.

In [26]:
%timeit large_df["amount"].apply(lambda x: x * 1.10)

79.2 ms ± 1.21 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Performance Interpretation

The `%timeit` results provide an empirical comparison between the two approaches.

The vectorized expression operates directly on the Pandas Series, while `.apply()` calls a Python lambda function for each value.

For this simple arithmetic operation, the vectorized approach should be faster than `.apply()`.

This demonstrates why vectorized operations should be preferred whenever the required transformation can be expressed directly using Pandas or NumPy operations.

`.apply()` remains useful for transformations that cannot reasonably be expressed using standard vectorized operations.

In [27]:
## Final cleaned dataset
df

,date,category,amount,customer_age,total_vectorized
0,2026-08-01,electronics,250.0,25.0,275.00
1,2026-08-02,electronics,180.0,31.0,198.00
2,2026-08-04,unknown,450.0,41.0,495.00
3,2026-08-05,clothing,120.0,29.0,132.00
4,2026-08-06,electronics,320.0,36.0,352.00
5,2026-08-07,furniture,275.0,31.0,302.50
6,2026-08-08,clothing,262.5,31.0,288.75


In [28]:
df.isna().sum()

date                0
category            0
amount              0
customer_age        0
total_vectorized    0
dtype: int64

In [29]:
df["category"].value_counts()

category
electronics    3
clothing       2
unknown        1
furniture      1
Name: count, dtype: int64

## Final Summary

This practice covered a complete Pandas data-cleaning and analysis workflow.

I first created a deliberately messy sales dataset containing missing values and inconsistent category names. I diagnosed the dataset using `head()`, `info()`, `shape`, `describe()`, and `isna().sum()` before making any changes.

Missing values were handled using different strategies for different columns. The missing transaction date was removed, while missing category, amount, and customer age values were filled using appropriate values.

I demonstrated the difference between `.loc` and `.iloc` and showed how sorting can cause label-based and position-based selection to return different rows.

I used Boolean filtering with `&` and parentheses and deliberately reproduced the error caused by using Python's `and` operator with Pandas Series.

The inconsistent category values were identified with `value_counts()` and standardized using a vectorized string operation.

Finally, I used `groupby()` to calculate mean sales amount, transaction count, and total sales amount per category. I also compared a vectorized calculation with an equivalent `.apply()` calculation using `%timeit` to demonstrate the performance advantage of vectorized operations.

The complete workflow was:

**Inspect → Diagnose → Decide → Clean → Verify → Analyze → Benchmark**

# Practice — Online Retail Dataset
This section applies the Pandas concepts learned above to a real-world retail transaction dataset.

Unlike the previous deliberately messy dataset, the problems in this dataset are not assumed in advance. I will first inspect and diagnose the data, identify actual data-quality issues, and then choose appropriate cleaning strategies based on the evidence.

The workflow will be:

**Load → Inspect → Diagnose → Clean → Verify → Analyze**

In [30]:
## Import required libraries
import pandas as pd
import numpy as np

## Load the Real Online Retail Dataset

The Online Retail dataset is a real-world transactional dataset containing sales records.

The dataset has been converted from Excel to CSV to make loading and processing more efficient with Pandas.

No cleaning is performed at this stage. The dataset is loaded as-is so that its original data quality can be diagnosed first.

In [34]:
retail_df = pd.read_csv("data/Online_Retail.csv")

retail_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Check Dataset Dimensions

The `shape` attribute returns the number of rows and columns in the DataFrame.

This provides an initial understanding of the size of the dataset before detailed inspection.

In [35]:
df.shape

(541909, 8)

## Inspect Column Names

I will inspect the column names to understand what information is available for each transaction.

In [36]:
retail_df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

## Inspect Dataset Information

Before making any cleaning decisions, I will inspect the structure of the dataset using `info()`.

This shows the column names, number of non-null values, data types, and memory usage. It helps identify missing values and columns whose data types may need attention.

In [37]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## Statistical Summary

The `describe()` method provides summary statistics for numerical columns.

I will use these statistics to identify unusual values such as negative quantities, zero prices, or unusually large transaction values before deciding how to handle them.

In [38]:
retail_df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## Diagnose Missing Values

Missing values are one of the most common data-quality problems in real-world datasets.

I will first calculate the number of missing values in every column. No values will be removed or filled until the extent of the missing data is understood.

In [39]:
retail_df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

### Missing Value Percentage

To better understand the impact of missing data, I will calculate the percentage of missing values in each column.

In [40]:
missing_percentage = (retail_df.isna().sum() / len(retail_df)) * 100

missing_percentage.sort_values(ascending=False)

CustomerID     24.926694
Description     0.268311
StockCode       0.000000
InvoiceNo       0.000000
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
Country         0.000000
dtype: float64

## Missing Data Findings

The dataset contains 541,909 transaction records.

The `CustomerID` column has 135,080 missing values, representing approximately 24.93% of the dataset. Since a missing customer identifier cannot be reliably reconstructed, these values should not be replaced with an artificial customer ID.

The `Description` column contains 1,454 missing values, representing approximately 0.27% of the dataset. This is a relatively small proportion, but the affected records should be inspected before deciding whether to remove them.

The remaining columns have no missing values.

Therefore, missing data will be handled based on the meaning and purpose of each column rather than using a blanket `dropna()` operation.

## Inspecting Missing Customer IDs

The `CustomerID` column contains a substantial number of missing values.

Before deciding how to handle these records, I will inspect examples of transactions where the customer identifier is missing. This helps determine whether these rows still contain useful transaction-level information.

In [41]:
retail_df[retail_df["Description"].isna()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [42]:
retail_df[retail_df["CustomerID"].isna()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


## Missing Data Cleaning Strategy

After inspecting the missing records, I will handle each column according to its meaning rather than applying one blanket rule.

### CustomerID

`CustomerID` has 135,080 missing values (approximately 24.93%). Some transactions with a missing customer ID still contain useful information such as product, quantity, price, date, and country.

I will not replace missing customer IDs with an artificial value because the actual customer cannot be reliably determined. These records can remain available for transaction-level analysis, while customer-level analysis should exclude rows where `CustomerID` is missing.

### Description

`Description` has 1,454 missing values (approximately 0.27%). The inspected records show that some of these transactions also have a zero unit price and missing customer information.

Because the missing descriptions cannot be reliably reconstructed from the available data, I will remove rows where `Description` is missing when performing product-level analysis.

This column-specific approach avoids unnecessarily deleting the large number of transactions that only have a missing `CustomerID`.

## Check for Duplicate Records

Duplicate records can cause transactions to be counted more than once during analysis.

I will first quantify duplicate rows before deciding whether any duplicates should be removed.

In [43]:
retail_df.duplicated().sum()

np.int64(5268)

## Duplicate Records

The dataset contains 5,268 completely duplicated rows.

Duplicate records can cause transactions to be counted more than once and may distort summary statistics and group-level analysis.

Before removing them, I will inspect a few duplicate records to confirm that they are exact duplicates.

In [44]:
retail_df[retail_df.duplicated(keep=False)].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [45]:
#Drop Duplicate records 
retail_df = retail_df.drop_duplicates().copy()

In [46]:
retail_df.duplicated().sum()

np.int64(0)

In [47]:
retail_df.shape

(536641, 8)

## Convert InvoiceDate to Datetime

The `InvoiceDate` column is currently stored as an `object` data type.

Since this column represents transaction date and time, it should be converted to Pandas `datetime` format. This will allow proper date-based filtering, sorting, and time-based analysis.

In [48]:
retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"])

retail_df["InvoiceDate"].dtype

dtype('<M8[ns]')

In [49]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  object        
 1   StockCode    536641 non-null  object        
 2   Description  535187 non-null  object        
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[ns]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 36.8+ MB


## Check for Invalid Quantity and UnitPrice Values

The `Quantity` and `UnitPrice` columns are important for calculating sales.

Before cleaning these columns, I will inspect their unusual values. In particular, negative quantities may represent returned or cancelled transactions, while zero or negative unit prices may represent non-standard records.

I will diagnose these values first instead of removing them without understanding their meaning.

In [50]:
retail_df[retail_df["Quantity"] < 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [51]:
(retail_df["Quantity"] < 0).sum()

np.int64(10587)

In [52]:
retail_df[retail_df["UnitPrice"] <= 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [53]:
(retail_df["UnitPrice"] <= 0).sum()

np.int64(2512)

In [54]:
print("Negative Quantity:", (retail_df["Quantity"] < 0).sum())
print("Zero/Negative UnitPrice:", (retail_df["UnitPrice"] <= 0).sum())

Negative Quantity: 10587
Zero/Negative UnitPrice: 2512


## Investigate Negative Quantities

Negative quantities may represent cancelled or returned transactions rather than invalid data.

In the Online Retail dataset, invoices beginning with `C` are associated with cancellations. Therefore, I will compare negative quantities with invoice numbers before deciding how to handle them.

In [55]:
retail_df[retail_df["Quantity"] < 0][
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
].head(10)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
141,C536379,D,Discount,-1,27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25


In [56]:
negative_qty = retail_df[retail_df["Quantity"] < 0]

negative_qty["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(9251)

In [57]:
negative_count = (retail_df["Quantity"] < 0).sum()

negative_invoice_count = (
    negative_qty["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

print("Negative quantity records:", negative_count)
print("Negative quantity with cancellation invoice:", negative_invoice_count)

Negative quantity records: 10587
Negative quantity with cancellation invoice: 9251


## Investigate Negative Quantities Without Cancellation Invoices

Most negative-quantity records are associated with invoice numbers beginning with `C`, which indicates cancellations.

However, 1,336 negative-quantity records do not have a cancellation invoice prefix. I will inspect these records separately before deciding whether they should be removed or preserved.

In [58]:
negative_non_cancelled = retail_df[
    (retail_df["Quantity"] < 0) &
    (~retail_df["InvoiceNo"].astype(str).str.startswith("C"))
]

negative_non_cancelled.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


In [60]:
negative_non_cancelled.shape

(1336, 8)

In [61]:
negative_non_cancelled["InvoiceNo"].head(20).tolist()

['536589',
 '536764',
 '536996',
 '536997',
 '536998',
 '537000',
 '537001',
 '537003',
 '537004',
 '537005',
 '537006',
 '537007',
 '537008',
 '537009',
 '537010',
 '537011',
 '537012',
 '537013',
 '537014',
 '537015']

## Handle Invalid Negative Quantity Records

The investigation showed that most negative-quantity transactions have cancellation invoice numbers beginning with `C`.

The remaining 1,336 negative-quantity records without a cancellation prefix also show characteristics such as missing descriptions, zero unit prices, and missing customer IDs in the inspected examples.

For the purpose of analyzing completed sales transactions, negative-quantity records will therefore be excluded. This prevents returns, cancellations, and non-standard records from distorting positive sales analysis.

In [62]:
# Remove transactions with negative quantities
retail_df = retail_df[retail_df["Quantity"] >= 0].copy()

print("Remaining rows:", len(retail_df))
print("Negative quantities:", (retail_df["Quantity"] < 0).sum())

Remaining rows: 526054
Negative quantities: 0


In [63]:
retail_df["Quantity"].describe()

count    526054.000000
mean         10.730874
std         157.591838
min           1.000000
25%           1.000000
50%           4.000000
75%          11.000000
max       80995.000000
Name: Quantity, dtype: float64

In [64]:
retail_df["UnitPrice"].describe()

count    526054.000000
mean          3.871747
std          42.016560
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       13541.330000
Name: UnitPrice, dtype: float64

In [65]:
(retail_df["UnitPrice"] <= 0).sum()

np.int64(1176)

In [66]:
retail_df[retail_df["UnitPrice"] <= 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
4348,536765,84952C,NaN,19,2010-12-02 14:43:00,0.0,NaN,United Kingdom


In [67]:
## Clean UnitPrice Column
# Remove records with zero or negative unit prices
retail_df = retail_df[retail_df["UnitPrice"] > 0].copy()

In [68]:
(retail_df["UnitPrice"] <= 0).sum()

np.int64(0)

In [69]:
retail_df.shape

(524878, 8)

In [71]:
retail_df.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
dtype: int64

### CustomerID Missing Values

After cleaning negative quantities and non-positive unit prices, `CustomerID`
still contains missing values.

These missing customer identifiers cannot be reliably reconstructed from the
available data. Therefore, they will not be replaced with artificial IDs.

The transactions will be retained because they still contain useful information
such as InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, and Country.

For customer-level analysis, rows with missing CustomerID will be excluded only
when a valid customer identifier is required.

In [72]:
retail_df.duplicated().sum()

np.int64(0)

In [73]:
retail_df.shape

(524878, 8)

In [81]:
## Final dataset information
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 524878 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    524878 non-null  object        
 1   StockCode    524878 non-null  object        
 2   Description  524878 non-null  object        
 3   Quantity     524878 non-null  int64         
 4   InvoiceDate  524878 non-null  datetime64[ns]
 5   UnitPrice    524878 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      524878 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 36.0+ MB


In [82]:

##Final missing values
retail_df.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
dtype: int64

In [83]:
## Negative quantity check
(retail_df["Quantity"] < 0).sum()

np.int64(0)

In [84]:
## Invalid UnitPrice check
(retail_df["UnitPrice"] <= 0).sum()

np.int64(0)

In [85]:
## Duplicate check
retail_df.duplicated().sum()

np.int64(0)

In [80]:
## Final statistical summary
retail_df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,524878.000000,524878,524878.000000,392692.000000
mean,10.616600,2011-07-04 15:30:16.317049088,3.922573,15287.843865
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000
25%,1.000000,2011-03-28 12:13:00,1.250000,13955.000000
50%,4.000000,2011-07-20 11:22:00,2.080000,15150.000000
75%,11.000000,2011-10-19 11:41:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,13541.330000,18287.000000
std,156.280031,NaN,36.093028,1713.539549


In [86]:

# Calculate Total Sales for Each Transaction

# Total sales value is calculated as:
# Quantity × UnitPrice
#
# This gives us the monetary value of each transaction.
retail_df["TotalSales"] = retail_df["Quantity"] * retail_df["UnitPrice"]


# Display the first few transactions to verify
# that the calculation was performed correctly.
retail_df[["Quantity", "UnitPrice", "TotalSales"]].head()

,Quantity,UnitPrice,TotalSales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [87]:

# Calculate Overall Sales

# Sum the TotalSales column to calculate
# the total revenue represented in the dataset.
total_sales = retail_df["TotalSales"].sum()

print("Total Sales:", total_sales)

Total Sales: 10642110.804


In [88]:

# Calculate Average Transaction Value

# Calculate the average sales amount per transaction.
average_transaction = retail_df["TotalSales"].mean()

print("Average Transaction Value:", average_transaction)

Average Transaction Value: 20.27539886221179
